Run before everything

In [4]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Part2

In [13]:
import time

# --- Configuration Constants ---
KP = 0.47          
BASE_SPEED = 24    
FINISH_SPEED = 18  
ZIGZAG_SPEED = 12 

def follow_line_step(kp, base_speed):
    """Executes one step of line following. All values forced to int."""
    offset, line_type, x, y = got.get_single_track_total_info()
    
    if line_type == 0:
        return None
    
    steering = int(offset * kp)
    speed_reduction = int(abs(steering * 0.25))
    current_speed = int(max(base_speed - speed_reduction, 12))
    
    got.mecanum_move_xyz(0, current_speed, steering)
    return line_type

def zigzag_search(direction):
    """Directional search using integer motor commands."""
    print(f"Zigzagging towards: {direction}")
    
    # Left = Positive Z | Right = Negative Z
    primary_z = 25 if direction == "LEFT" else -25
    secondary_z = -int(primary_z * 0.8) # Slightly smaller back-sweep
    
    found = False
    for i in range(1, 5): # Increased to 4 sweeps for better recovery
        duration = 0.4 * i
        
        # Sweep toward choice
        got.mecanum_move_xyz(0, int(ZIGZAG_SPEED), int(primary_z))
        s_time = time.time()
        while time.time() - s_time < duration:
            if got.get_single_track_total_info()[1] != 0:
                found = True; break
            time.sleep(0.01)
        if found: break

        # Sweep back
        got.mecanum_move_xyz(0, int(ZIGZAG_SPEED), int(secondary_z))
        s_time = time.time()
        while time.time() - s_time < (duration * 1.2):
            if got.get_single_track_total_info()[1] != 0:
                found = True; break
            time.sleep(0.01)
        if found: break
        
    if found:
        got.mecanum_stop()
        time.sleep(0.1)
    return found

def part2run():
    print("Mission Start")
    got.set_track_recognition_line(0) 

    # --- PHASE 0: Initial Acquisition ---
    if got.get_single_track_total_info()[1] == 0:
        print("Searching for start line...")
        got.mecanum_move_xyz(0, 12, 0)
        while got.get_single_track_total_info()[1] == 0:
            time.sleep(0.01)

    # --- PHASE 1: Initial Path ---
    print("Following to Fork...")
    while True:
        status = follow_line_step(KP, BASE_SPEED)
        if status is None:
            got.mecanum_stop()
            break
        time.sleep(0.01)

    # --- PHASE 2: OCR Recognition (Floor or Sign) ---
    print("Scanning for Direction...")
    choice = ""
    while not choice:
        # result is converted to string to avoid attribute errors
        word = str(got.get_words_result()).upper()
        
        if any(p in word for p in ["LEFT", "LEF", "EFT", "LFT"]):
            choice = "LEFT"
        elif any(p in word for p in ["RIGHT", "RIG", "GHT", "RGT"]):
            choice = "RIGHT"
        
        # If no word is seen, robot stays still to get a clear image
        got.mecanum_stop()
        time.sleep(0.1)
    
    print(f"Path Confirmed: {choice}")
    got.screen_display_background(6)

    # --- PHASE 3: The 45-Degree Turn & Search ---
    print(f"Executing 45-degree {choice} turn...")
    
    # 1. Nudge forward to get wheels into the intersection center
    got.mecanum_move_xyz(0, -10, 0)
    time.sleep(0.5)
    
    # 2. Perform the 45-degree angle turn (Adjust sleep for exact 45 deg)
    turn_z = 55 if choice == "LEFT" else -55
    got.mecanum_move_xyz(0, 10, int(turn_z))
    time.sleep(0.7) # Approx time for 45 degrees at speed 10/25
    
    # 3. Move forward slightly into the new branch area
    got.mecanum_move_xyz(0, 15, 0)
    time.sleep(0.4)
    got.mecanum_stop()

    # 4. Start Zigzag Search to lock onto the line
    if not zigzag_search(choice):
        print("Line not found in zigzag, trying pivot recovery...")
        got.mecanum_move_xyz(0, 0, int(turn_z))
        while got.get_single_track_total_info()[1] == 0:
            time.sleep(0.01)

    # --- PHASE 4: Finish Line Trace ---
    print("Final Trace to Finish...")
    while True:
        status = follow_line_step(KP, FINISH_SPEED)
        if status is None:
            time.sleep(0.3) # Debounce
            if got.get_single_track_total_info()[1] == 0:
                got.mecanum_stop()
                got.screen_print("FINISHED")
                print("Goal Reached.")
                break
        time.sleep(0.01)

if __name__ == "__main__":
    try:
        part2run()
    except Exception as e:
        print(f"System Error: {e}")
        got.mecanum_stop()

Mission Start
Following to Fork...
Scanning for Direction...
Path Confirmed: LEFT
Executing 45-degree LEFT turn...
Zigzagging towards: LEFT
Final Trace to Finish...
System Error: 'UGOT' object has no attribute 'screen_print'
